In [1]:
%pip install langchain-groq python-dotenv

  Using cached langchain_groq-1.1.3-py3-none-any.whl.metadata (2.9 kB)
  Using cached groq-0.37.1-py3-none-any.whl.metadata (16 kB)
Using cached langchain_groq-1.1.3-py3-none-any.whl (20 kB)
Using cached groq-0.37.1-py3-none-any.whl (137 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-groq]

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
print(os.getcwd())

/Users/aaranachaurasia/earnings-call-analyst/notebooks


In [4]:
os.chdir("..")
print(os.getcwd())

/Users/aaranachaurasia/earnings-call-analyst


In [6]:
print(os.path.exists(".env"))

True


In [7]:
os.listdir(".")

['.DS_Store',
 'app',
 'requirements.txt',
 'Dockerfile',
 'models',
 'README.md',
 '.gitignore',
 '.env',
 'docker-compose.yml',
 'chroma_db',
 'eval',
 'venv',
 'data',
 'notebooks']

In [9]:
with open(".env", "w") as f:
    f.write("GROQ_API_KEY=YOUR_API_KEY_HERE\n")

In [10]:
from dotenv import load_dotenv
load_dotenv()
print(os.getenv("GROQ_API_KEY")[:8])

gsk_SPKr


In [11]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

In [12]:
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path="./notebooks/chroma_db")
collection = client.get_or_create_collection(name="earnings_calls")

print(f"Collection has {collection.count()} documents")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7967.79it/s]


Collection has 0 documents


In [13]:
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="earnings_calls")

print(f"Collection has {collection.count()} documents")

Collection has 146 documents


In [14]:
def retrieve_chunks(question, n_results=5):
    query_embedding = embeddings.embed_query(question)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )
    return results["documents"][0], results["metadatas"][0]

# test it
docs, metas = retrieve_chunks("What did JPMorgan say about net interest income?")
for doc, meta in zip(docs, metas):
    print(f"[{meta['company']} {meta['quarter']} {meta['year']}]: {doc[:150]}...")
    print()

[JPMorgan Q1 2024]: inflows and higher average market levels, partially offset by lower NII due to deposit
×
()
8/11/26, 2:58 PM
JPMorgan Chase (JPM) Q1 2024 Earnings Cal...

[JPMorgan Q1 2024]: distribution. This quarter's higher RWA is largely due to seasonal effects, including higher
client activity in Markets and higher risk weights on def...

[JPMorgan Q1 2024]: was encouraging to see some positive momentum in announced M&A in the quarter, it
×
()
8/11/26, 2:58 PM
JPMorgan Chase (JPM) Q1 2024 Earnings Call Tra...

[JPMorgan Q1 2024]: about 3.5% above the effective tax rate.
×
()
8/11/26, 2:58 PM
JPMorgan Chase (JPM) Q1 2024 Earnings Call Transcript | The Motley Fool
https://www.foo...

[JPMorgan Q1 2024]: at JPMorgan and me personally. I'm thrilled to have you on this call. For those who don't
know, Betsy has been through a terrible medical episode.
And...



In [15]:
def generate_answer(question, n_results=5):
    docs, metas = retrieve_chunks(question, n_results)
    
    context = "\n\n".join([
        f"[{meta['company']} {meta['quarter']} {meta['year']}]: {doc}"
        for doc, meta in zip(docs, metas)
    ])
    
    system_prompt = """You are a strict equity research analyst. Answer ONLY using the provided context below.
If comparing companies, structure your answer clearly per company.
If a specific number or figure is not in the context, say "Not mentioned in transcript" — never invent numbers.
Always mention which company and quarter your answer is drawn from."""

    user_prompt = f"""Context:
{context}

Question: {question}"""

    response = llm.invoke([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ])
    
    return response.content

# test it
answer = generate_answer("What did JPMorgan say about net interest income in Q1 2024?")
print(answer)

JPMorgan (JPM) in Q1 2024: 
The company mentioned that the journey toward NII (Net Interest Income) normalization begins, but it also mentioned lower NII due to deposit margin compression. However, specific NII figures for Q1 2024 are Not mentioned in transcript.


In [16]:
def classify_question(question):
    q_lower = question.lower()
    
    comparison_keywords = ["compare", "vs", "versus", "difference between"]
    temporal_keywords = ["changed", "over time", "trend", "from q1 to q2", "quarter over quarter"]
    
    if any(kw in q_lower for kw in comparison_keywords):
        return "comparison"
    elif any(kw in q_lower for kw in temporal_keywords):
        return "temporal"
    else:
        return "single"

# test it on your 3 roadmap questions
test_questions = [
    "What did JPMorgan say about net interest income in Q1 2024?",
    "Compare how JPMorgan and HDFC described credit risk in Q1 2024",
    "How did JPMorgan's outlook change from Q1 to Q2 2024?"
]

for q in test_questions:
    print(f"{classify_question(q)}: {q}")

single: What did JPMorgan say about net interest income in Q1 2024?
comparison: Compare how JPMorgan and HDFC described credit risk in Q1 2024
temporal: How did JPMorgan's outlook change from Q1 to Q2 2024?


In [17]:
def extract_companies(question):
    """Simple keyword-based company extraction — checks which known companies are mentioned."""
    known_companies = ["JPMorgan", "HDFC", "Infosys"]
    q_lower = question.lower()
    found = [c for c in known_companies if c.lower() in q_lower]
    return found

def retrieve_with_routing(question, n_results=5):
    question_type = classify_question(question)
    companies = extract_companies(question)
    
    if question_type == "single":
        # one filtered search if a company is mentioned, else unfiltered
        query_embedding = embeddings.embed_query(question)
        where_filter = {"company": companies[0]} if companies else None
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
            where=where_filter
        )
        return results["documents"][0], results["metadatas"][0]
    
    elif question_type == "comparison":
        # two separate filtered searches, one per company, combined
        all_docs, all_metas = [], []
        query_embedding = embeddings.embed_query(question)
        for company in companies:
            results = collection.query(
                query_embeddings=[query_embedding],
                n_results=n_results,
                where={"company": company}
            )
            all_docs.extend(results["documents"][0])
            all_metas.extend(results["metadatas"][0])
        return all_docs, all_metas
    
    elif question_type == "temporal":
        # search across quarters for the mentioned company, no quarter filter
        query_embedding = embeddings.embed_query(question)
        where_filter = {"company": companies[0]} if companies else None
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results * 2,  # more results since spanning quarters
            where=where_filter
        )
        return results["documents"][0], results["metadatas"][0]

In [18]:
def generate_answer_v2(question, n_results=5):
    docs, metas = retrieve_with_routing(question, n_results)
    
    context = "\n\n".join([
        f"[{meta['company']} {meta['quarter']} {meta['year']}]: {doc}"
        for doc, meta in zip(docs, metas)
    ])
    
    system_prompt = """You are a strict equity research analyst. Answer ONLY using the provided context below.
If comparing companies, structure your answer clearly per company.
If a specific number or figure is not in the context, say "Not mentioned in transcript" — never invent numbers.
Always mention which company and quarter your answer is drawn from."""

    user_prompt = f"""Context:
{context}

Question: {question}"""

    response = llm.invoke([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ])
    
    return response.content

# test the comparison question — this is the important one
answer = generate_answer_v2("Compare how JPMorgan and HDFC described credit risk in Q1 2024")
print(answer)

**JPMorgan (Q1 2024)**
JPMorgan described credit risk as being managed through various skills such as underwriting, structuring, origination, distribution, secondary trading, risk appetite, and credit analysis capabilities. They mentioned that people are behaving rationally and managing their balance sheets in a post-pandemic type of way, which is good news from a credit perspective. They also noted that corporate lending spreads have widened, indicating some disciplining of lending.

**HDFC (Q1 2024)**
HDFC described credit risk in terms of their credit cost ratio, which was at 70 basis points for the quarter, and net of recoveries, it was at 51 basis points. They also mentioned that their gross NPA ratio was at 1.17%, and their provision coverage ratio was at 66%. HDFC noted that their credit risk is managed through their risk management framework, which evolves over time and takes into account the maturity of their loan book and the overall economic cycle.

Overall, both banks seem 

In [19]:
answer = generate_answer_v2("How did JPMorgan's outlook change from Q1 to Q2 2024?")
print(answer)

Not mentioned in transcript. The provided context only discusses JPMorgan's Q1 2024 earnings call and does not mention Q2 2024.


In [20]:
import fitz
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

def parse_filename(filepath):
    filename = os.path.basename(filepath)
    name = filename.replace(".pdf", "")
    parts = name.split("_")
    return {"company": parts[0], "market": parts[1], "year": int(parts[2]), "quarter": parts[3]}

In [21]:
def ingest_pdf(filepath):
    doc = fitz.open(filepath)
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    
    chunks = text_splitter.split_text(full_text)
    metadata = parse_filename(filepath)
    
    texts, metadatas, ids = [], [], []
    for i, chunk in enumerate(chunks):
        chunk_metadata = metadata.copy()
        chunk_metadata["section"] = "unclassified"
        texts.append(chunk)
        metadatas.append(chunk_metadata)
        ids.append(f"{metadata['company']}_{metadata['quarter']}_{metadata['year']}_chunk{i}")
    
    embedded_vectors = embeddings.embed_documents(texts)
    collection.add(documents=texts, embeddings=embedded_vectors, metadatas=metadatas, ids=ids)
    print(f"Ingested {filepath}: {len(chunks)} chunks added")

# ingest the remaining 4 files
remaining_files = [
    "data/US/JPMorgan/JPMorgan_US_2024_Q2.pdf",
    "data/India/HDFC/HDFC_India_2024_Q2.pdf",
    "data/India/Infosys/Infosys_India_2024_Q1.pdf",
    "data/India/Infosys/Infosys_India_2024_Q2.pdf"
]

for filepath in remaining_files:
    ingest_pdf(filepath)

print(f"\nTotal documents now: {collection.count()}")

Ingested data/US/JPMorgan/JPMorgan_US_2024_Q2.pdf: 75 chunks added
Ingested data/India/HDFC/HDFC_India_2024_Q2.pdf: 60 chunks added
Ingested data/India/Infosys/Infosys_India_2024_Q1.pdf: 58 chunks added
Ingested data/India/Infosys/Infosys_India_2024_Q2.pdf: 56 chunks added

Total documents now: 395


In [22]:
answer = generate_answer_v2("How did JPMorgan's outlook change from Q1 to Q2 2024?")
print(answer)

**JPMorgan Chase (JPM) - Q1 2024 and Q2 2024**

The outlook for JPMorgan Chase changed from Q1 to Q2 2024 in the following ways:

* In Q1 2024, the company mentioned that the economic, geopolitical, and regulatory uncertainties remained prominent. 
* In Q2 2024, Jeremy Barnum stated that the dialogue on ECM is elevated, and the dialogue on M&A is quite robust, which encourages them and makes them hopeful that they could be seeing a better trend in the Investment Banking space.

However, specific details about the overall outlook change are Not mentioned in transcript.
